In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_validate, GroupKFold, LeaveOneGroupOut
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,accuracy_score
import statsmodels.api as sm
from statsmodels.tools.eval_measures import mse, rmse
import unicodedata
import re
import os

In [2]:
# 1. Cargar los datos
BASE_DIR = "."

#ruta_df = os.path.join(BASE_DIR, "df_base_rendimiento_clima_cafe.csv")
ruta_df = os.path.join(BASE_DIR, "Base_data_fe_pca_reduction.csv")

df_total = pd.read_csv(ruta_df)

In [3]:
# 2. Definir función para particionar los datos por año y estimar con base en los años seleccionados a predecir
def logo_seleccion_anio(data, feature_cols, target_col, year_col, min_train_years=10):
    """
    For each year, train on all previous years and predict that year
    """
    years = sorted(data[year_col].unique())
    results = []
    
    
    
    for i, test_year in enumerate(years):
        # Proponemos un limite minimo de datos para trabajar
        if i < min_train_years:
            print(f"Saltamos {test_year}: no hay datos suficientes\n")
            continue
        
        # Training years = todos los años antes de test_year
        train_years = years[:i]
        
        # =========================
        # 2. Train / test
        # =========================
        
        train_mask = data[year_col].isin(train_years)
        test_mask = data[year_col] == test_year
        
        X_train = data.loc[train_mask, feature_cols]
        y_train = data.loc[train_mask, target_col]
        X_test = data.loc[test_mask, feature_cols]
        y_test = data.loc[test_mask, target_col]

        # =========================
        # 3. Preprocesamiento
        # =========================

        num_cols = ['humedad relativa media anual (%)', 'humedad volumétrica media anual del suelo capa 1 (m³/m³)',
                'spi6_meses_bajo_m1', 'spi12_mean_anual','spi12_meses_bajo_m1','spi3_dic','spi12_dic','spi1_floracion',
                'altitud_media_m','amplitud_temperatura_media_mensual','balance_precipitacion_evaporacion',
                'log1p_área sembrada','log1p_radiación solar acumulada anual (mj/m²/año)','humedad_suelo_diferencia_capa2_capa1',
                'spi3_presion_sequia_anual','spi1_cambio_llenado_vs_floracion','spi3_cambio_llenado_vs_floracion',
                ]
        
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]), num_cols)
            ]
        )
        
        X_train_processed = preprocessor.fit_transform(X_train)
        X_test_processed = preprocessor.transform(X_test)

        # =========================
        # 4. Modelos
        # =========================
        
        # Añadir constantes para el model OLS
        X_train_with_const = sm.add_constant(X_train_processed)
        X_test_with_const = sm.add_constant(X_test_processed)
        
        # Entrenar modelo
        ols_model = sm.OLS(y_train, X_train_with_const).fit()

        # =========================
        # 5. Predicción y evaluación
        # =========================
        
        predictions = ols_model.predict(X_test_with_const)
        
        mse_value = mse(y_test, predictions)
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        
        results.append({
            'test_year': test_year,
            'train_years': train_years,
            'n_train_samples': len(X_train),
            'n_test_samples': len(X_test),
            'mse': mse_value,
            'mae': mae,
            'r2': r2,
        })

        column_names = ['const'] + list(feature_cols)

        coef_table = pd.DataFrame({
            'Variable': column_names,
            'Coeficiente': ols_model.params,
            'Std. Error': ols_model.bse,
            't': ols_model.tvalues,
            'P>|t|': ols_model.pvalues,
            'CI_lower': ols_model.conf_int()[0],
            'CI_upper': ols_model.conf_int()[1]
        })
        
        coef_table['Signif'] = coef_table['P>|t|'].apply(
            lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ('.' if p < 0.1 else '')))
        )
        
        coef_table = coef_table[['Variable', 'Coeficiente', 'Std. Error', 't', 'P>|t|','Signif', 'CI_lower', 'CI_upper']]

        
        print(f"Train on {train_years} → Predict {test_year}: MAE={mae:.4f}, R²={r2:.4f}")
    
    return results,coef_table


In [4]:
# Run rolling window LOGO
feature_cols = ['humedad relativa media anual (%)', 'humedad volumétrica media anual del suelo capa 1 (m³/m³)',
                'spi6_meses_bajo_m1', 'spi12_mean_anual','spi12_meses_bajo_m1','spi3_dic','spi12_dic','spi1_floracion',
                'altitud_media_m','amplitud_temperatura_media_mensual','balance_precipitacion_evaporacion',
                'log1p_área sembrada','log1p_radiación solar acumulada anual (mj/m²/año)','humedad_suelo_diferencia_capa2_capa1',
                'spi3_presion_sequia_anual','spi1_cambio_llenado_vs_floracion','spi3_cambio_llenado_vs_floracion',
                ]


results_df,coef_table = logo_seleccion_anio(df_total, feature_cols, 'rendimiento', 'anio', min_train_years=10)
print("\n" + "="*50)
print("Resultados:")
print(pd.DataFrame(results_df))
print(coef_table)

Saltamos 2007: no hay datos suficientes

Saltamos 2008: no hay datos suficientes

Saltamos 2009: no hay datos suficientes

Saltamos 2010: no hay datos suficientes

Saltamos 2011: no hay datos suficientes

Saltamos 2012: no hay datos suficientes

Saltamos 2013: no hay datos suficientes

Saltamos 2014: no hay datos suficientes

Saltamos 2015: no hay datos suficientes

Saltamos 2016: no hay datos suficientes

Train on [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016)] → Predict 2017: MAE=0.2391, R²=-0.4189
Train on [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)] → Predict 2018: MAE=0.2694, R²=-0.5204
Train on [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np

#### Ajuste Mejor modelo
Con base en el R2 el mejor modelo es el que predice el año 2024 con base en el resto de años (2007 al 2023)

In [15]:
feature_cols = ['humedad relativa media anual (%)', 'humedad volumétrica media anual del suelo capa 1 (m³/m³)',
                'spi6_meses_bajo_m1', 'spi12_mean_anual','spi12_meses_bajo_m1','spi3_dic','spi12_dic','spi1_floracion',
                'altitud_media_m','amplitud_temperatura_media_mensual','balance_precipitacion_evaporacion',
                'log1p_área sembrada','log1p_radiación solar acumulada anual (mj/m²/año)','humedad_suelo_diferencia_capa2_capa1',
                'spi3_presion_sequia_anual','spi1_cambio_llenado_vs_floracion','spi3_cambio_llenado_vs_floracion',
                ]


target_col='rendimiento'

results = []
years = sorted(df_total['anio'].unique())

# Training years = todos los años antes de test_year
# seleccionamos todos los años menos 2024
train_years = years[:-1]

# =========================
# 2. Train / test
# =========================
        
train_mask = df_total['anio'].isin(train_years)
test_mask = df_total['anio'] == 2024
        
X_train = df_total.loc[train_mask, feature_cols]
y_train = df_total.loc[train_mask, target_col]
X_test = df_total.loc[test_mask, feature_cols]
y_test = df_total.loc[test_mask, target_col]

# =========================
# 3. Preprocesamiento
# =========================

num_cols = ['humedad relativa media anual (%)', 'humedad volumétrica media anual del suelo capa 1 (m³/m³)',
            'spi6_meses_bajo_m1', 'spi12_mean_anual','spi12_meses_bajo_m1','spi3_dic','spi12_dic','spi1_floracion',
            'altitud_media_m','amplitud_temperatura_media_mensual','balance_precipitacion_evaporacion',
            'log1p_área sembrada','log1p_radiación solar acumulada anual (mj/m²/año)','humedad_suelo_diferencia_capa2_capa1',
            'spi3_presion_sequia_anual','spi1_cambio_llenado_vs_floracion','spi3_cambio_llenado_vs_floracion',
            ]
        
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols)
    ]
)
        
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# =========================
# 4. Modelos
# =========================
        
# Añadir constantes para el model OLS
X_train_with_const = sm.add_constant(X_train_processed)
X_test_with_const = sm.add_constant(X_test_processed)
        
# Entrenar modelo
ols_model = sm.OLS(y_train, X_train_with_const).fit()

# =========================
# 5. Predicción y evaluación
# =========================
        
predictions = ols_model.predict(X_test_with_const)
        
mse_value = mse(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
        
results.append({
    'test_year': 2024,
    'train_years': train_years,
    'n_train_samples': len(X_train),
    'n_test_samples': len(X_test),
    'mse': mse_value,
    'mae': mae,
    'r2': r2,
})

column_names = ['const'] + list(feature_cols)

coef_table = pd.DataFrame({
    'Variable': column_names,
    'Coeficiente': ols_model.params,
    'Std. Error': ols_model.bse,
    't': ols_model.tvalues,
    'P>|t|': ols_model.pvalues,
    'CI_lower': ols_model.conf_int()[0],
    'CI_upper': ols_model.conf_int()[1]
})
        
coef_table['Signif'] = coef_table['P>|t|'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ('.' if p < 0.1 else '')))
)
        
coef_table = coef_table[['Variable', 'Coeficiente', 'Std. Error', 't', 'P>|t|','Signif', 'CI_lower', 'CI_upper']]


Train on [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] → Predict 2024: MAE=0.2908, R²=-0.0956


In [16]:
# Metricas
results

[{'test_year': 2024,
  'train_years': [np.int64(2007),
   np.int64(2008),
   np.int64(2009),
   np.int64(2010),
   np.int64(2011),
   np.int64(2012),
   np.int64(2013),
   np.int64(2014),
   np.int64(2015),
   np.int64(2016),
   np.int64(2017),
   np.int64(2018),
   np.int64(2019),
   np.int64(2020),
   np.int64(2021),
   np.int64(2022),
   np.int64(2023)],
  'n_train_samples': 1506,
  'n_test_samples': 89,
  'mse': np.float64(0.1397061620526542),
  'mae': 0.29080657379421354,
  'r2': -0.09558641822172698}]

In [17]:
coef_table

,Variable,Coeficiente,Std. Error,t,P>|t|,Signif,CI_lower,CI_upper
const,const,0.956301,0.016199,59.034209,0.000000,***,0.924525,0.988076
x1,humedad relativa media anual (%),-0.013289,0.028204,-0.471182,0.637580,,-0.068613,0.042035
x2,humedad volumétrica media anual del suelo capa...,0.020764,0.021960,0.945517,0.344549,,-0.022313,0.063841
x3,spi6_meses_bajo_m1,0.001227,0.031727,0.038684,0.969148,,-0.061007,0.063461
x4,spi12_mean_anual,0.031421,0.034394,0.913551,0.361101,,-0.036045,0.098887
x5,spi12_meses_bajo_m1,0.079823,0.025428,3.139252,0.001727,**,0.029946,0.129701
x6,spi3_dic,0.123231,0.043487,2.833732,0.004663,**,0.037928,0.208534
x7,spi12_dic,0.101831,0.032752,3.109149,0.001912,**,0.037586,0.166077
x8,spi1_floracion,-0.088238,0.034283,-2.573811,0.010155,*,-0.155486,-0.020990
x9,altitud_media_m,-0.057890,0.025672,-2.255002,0.024278,*,-0.108247,-0.007533
